In [49]:
import pdfplumber
import pandas as pd
import re
import os

In [50]:
pdf_path = 'C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/biometrias_identificadas/15177_CRISTIANE RAMOS_MORAIS_15177_DOB19780710_AstigPlan_20240812174158.pdf'

In [51]:
def classificar_arquivos_por_tamanho(pasta):
    # Conversão para bytes
    limite_pequeno = 100 * 1024      # 100 KB
    limite_medio = 1000 * 1024       # 1000 KB (1 MB)

    # Contadores
    pequenos = 0
    medios = 0
    grandes = 0

    # Itera pelos arquivos da pasta
    for nome_arquivo in os.listdir(pasta):
        caminho_arquivo = os.path.join(pasta, nome_arquivo)
        if os.path.isfile(caminho_arquivo):
            tamanho = os.path.getsize(caminho_arquivo)
            if tamanho <= limite_pequeno:
                pequenos += 1
            elif tamanho <= limite_medio:
                medios += 1
            else:
                grandes += 1

    print(f"Arquivos até 100 KB: {pequenos}")
    print(f"Arquivos até 1000 KB: {medios}")
    print(f"Arquivos acima de 1000 KB: {grandes}")


        

In [52]:
from pdfminer.high_level import extract_text
import pandas as pd
import os

def find_r1_lines(lines,mat):
    r1_indices = []
    for idx, line in enumerate(lines):
        if line.strip().startswith(mat):
            r1_indices.append(idx)
    return r1_indices

def extract_biometric_data(pdf_path):
    full_text = extract_text(pdf_path)
    lines = full_text.split('\n')

    flat_index = find_r1_lines(lines, 'R1[mm/D/°]')
    step_index = find_r1_lines(lines, 'R2[mm/D/°]')
    axial_index = find_r1_lines(lines, 'AL [mm]')
    acd_index = find_r1_lines(lines, 'ACD [mm]')
    lens_thickness_index = find_r1_lines(lines, 'LT [mm]')
    wtw_index = find_r1_lines(lines, 'WTW [mm]')
    OD = {
        'name' : lines[5].split(',')[0].strip(),
        'olho_operado': 'OD',
        'k1':(
            lines[flat_index[0]].split('/')[-1].split('@')[0].strip().replace(',','.')
            if len(lines[flat_index[0]].split('@')) >= 2 and '@' in lines[step_index[0]] else ''),
        'eixo_plano':(
            lines[flat_index[0]].split('/')[-1].split('@')[1].strip().replace(',','.')
            if len(lines[flat_index[0]].split('@')) >= 2 and '@' in lines[step_index[0]] else ''),
        'k2':(lines[step_index[0]].split('/')[-1].split('@')[0].strip().replace(',','.')
            if len(lines[step_index[0]].split('@')) >= 2 and '@' in lines[step_index[0]] else ''
            ),
        'eixo_curvo':(lines[step_index[0]].split('/')[-1].split('@')[1].strip().replace(',','.')
            if len(lines[step_index[0]].split('@')) >= 2 and '@' in lines[step_index[0]] else ''
            ),
        'al':(lines[axial_index[0]].split('[mm]')[1].strip().replace(',','.')),
        'acd':(lines[acd_index[0]].split('[mm]')[1].strip().replace(',','.')),
        'lt':(lines[lens_thickness_index[0]].split('[mm]')[1].strip().replace(',','.')),
        'wtw':(lines[wtw_index[0]].split('[mm]')[1].strip().replace(',','.'))
        }
    OS = {
        'nome' : lines[5].split(',')[0].strip(),
        'olho_operado': 'OS',
        'k1':(
            lines[flat_index[1]].split('/')[-1].split('@')[0].strip().replace(',','.')
            if len(lines[flat_index[1]].split('@')) >= 2 and '@' in lines[flat_index[1]] else ''),
        'eixo_plano':(
            lines[flat_index[1]].split('/')[-1].split('@')[1].strip().replace(',','.')
            if len(lines[flat_index[1]].split('@')) >= 2 and '@' in lines[flat_index[1]] else ''
            ),
        'k2':(lines[step_index[1]].split('/')[-1].split('@')[0].strip().replace(',','.')
            if len(lines[step_index[1]].split('@')) >= 2 and '@' in lines[step_index[1]] else ''
            ),
        'eixo_curvo':(lines[step_index[1]].split('/')[-1].split('@')[1].strip().replace(',','.')
            if len(lines[step_index[1]].split('@')) >= 2 and ('@' in lines[step_index[1]]) else ''
            ),
        'al':(lines[axial_index[1]].split('[mm]')[1].strip().replace(',','.')),
        'acd':(lines[acd_index[1]].split('[mm]')[1].strip().replace(',','.')),
        'lt':(lines[lens_thickness_index[1]].split('[mm]')[1].strip().replace(',','.')),
        'wtw':(lines[wtw_index[1]].split('[mm]')[1].strip().replace(',','.'))
        }
    return [OD, OS]

In [53]:
pdf_path = 'C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/biometrias_identificadas/15090_MONTE_MARIA_DAS_GRACAS_1961-09-24_IOL.pdf'

In [54]:
extract_biometric_data(pdf_path)

[{'name': 'MONTE MARIA DAS GRACAS',
  'olho_operado': 'OD',
  'k1': '43.93',
  'eixo_plano': '42',
  'k2': '44.84',
  'eixo_curvo': '132',
  'al': '22.60',
  'acd': '2.80',
  'lt': '4.17',
  'wtw': '11.78'},
 {'nome': 'MONTE MARIA DAS GRACAS',
  'olho_operado': 'OS',
  'k1': '43.89',
  'eixo_plano': '109',
  'k2': '44.17',
  'eixo_curvo': '19',
  'al': '22.60',
  'acd': '2.74',
  'lt': '4.15',
  'wtw': '11.83'}]

In [55]:
df_pentacam = pd.read_excel("C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/Pacientes_Torica_com_dados_pentacam.xlsx")

In [56]:
print(df_pentacam.columns)

Index(['pront', 'nome', 'sexo', 'olho_operado', 'k1', 'eixo_plano', 'k2',
       'eixo_curvo', 'al', 'acd', 'lt', 'wtw', 'astig_refracional_pré_op',
       'astig_anterior_topo_pré_op', 'astig_posterior_topo_pré',
       'astig_refracional_pós_op', 'classificação_astigmatismo',
       'esferico_pre', 'cilindrico_pre', 'eixo_pre', 'esferico_pos',
       'cilindrico_pos', 'eixo_pos', 'topografia', 'valor', 'eixo',
       'tipo_padronizado', 'tipo_irregular',
       'tipo_regular_assimetrico_a_favor_da_regra',
       'tipo_regular_assimetrico_contra_a_regra',
       'tipo_regular_assimetrico_obliquo',
       'tipo_regular_simetrico_a_favor_da_regra',
       'tipo_regular_simetrico_contra_a_regra',
       'tipo_regular_simetrico_obliquo', 'anterior_rp', 'anterior_rc',
       'anterior_rm', 'anterior_k1', 'anterior_k2', 'anterior_km',
       'anterior_Eixo_plano', 'anterior_ast', 'anterior_rper', 'anterior_rmin',
       'posterior_rp', 'posterior_rc', 'posterior_rm', 'posterior_k1',
       

In [57]:
def find_exam_file(prontuario, folder_path):
    """
    Procura o arquivo do exame correspondente ao prontuário.
    Retorna o caminho completo do arquivo.
    """
    for filename in os.listdir(folder_path):
        if filename.startswith(str(prontuario)):
            return os.path.join(folder_path, filename)
    return None

In [58]:
results = []

for i, row in df_pentacam.iterrows():
    pront = row['pront']
    olho_df = row['olho_operado']  # 'OD' ou 'OS'

    # 1️⃣ Encontrar o arquivo do exame
    file_path = find_exam_file(pront, "C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/biometrias_identificadas/")

    if file_path is None:
        print(f"⚠️ Exame não encontrado para {pront} ({olho_df})")
        results.append({})
        continue

    # 2️⃣ Verificar tamanho do arquivo
    if os.path.getsize(file_path) < 100 * 1024:  # <100kb
        extracted_data = extract_biometric_data(file_path)  # sua função que retorna lista de dicts

        # 3️⃣ Encontrar o dict que corresponde ao olho do dataframe
        data_olho = next((d for d in extracted_data if d['olho_operado'] == olho_df), None)

        if data_olho is not None:
            # 4️⃣ Atualizar o dataframe linha a linha
            for col in ['k1', 'eixo_plano', 'k2', 'eixo_curvo', 'al', 'acd', 'lt', 'wtw']:
                df_pentacam.at[i, col] = data_olho.get(col)
            
            results.append(data_olho)
        else:
            print(f"⚠️ Dados não encontrados para {pront} ({olho_df}) no arquivo")
            results.append({})
    else:
        print(f"⚠️ Arquivo muito grande para {pront} ({olho_df})")
        results.append({})

⚠️ Arquivo muito grande para 121476 (OD)


C:\Users\luise\AppData\Local\Temp\ipykernel_19804\3804033791.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '40.67' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_pentacam.at[i, col] = data_olho.get(col)
C:\Users\luise\AppData\Local\Temp\ipykernel_19804\3804033791.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_pentacam.at[i, col] = data_olho.get(col)
C:\Users\luise\AppData\Local\Temp\ipykernel_19804\3804033791.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '42.75' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_pentacam.at[i, col] = data_o

⚠️ Exame não encontrado para 65154 (OD)
⚠️ Exame não encontrado para 65154 (OS)
⚠️ Arquivo muito grande para 105701 (OS)
⚠️ Arquivo muito grande para 48439 (OD)
⚠️ Arquivo muito grande para 48439 (OS)
⚠️ Exame não encontrado para 38485 (OD)
⚠️ Exame não encontrado para 38485 (OS)
⚠️ Arquivo muito grande para 34912 (OD)
⚠️ Arquivo muito grande para 34912 (OS)
⚠️ Arquivo muito grande para 5178 (OD)
⚠️ Arquivo muito grande para 15430 (OD)
⚠️ Arquivo muito grande para 15430 (OS)
⚠️ Exame não encontrado para 110982 (OD)
⚠️ Arquivo muito grande para 118848 (OD)
⚠️ Arquivo muito grande para 118848 (OS)
⚠️ Arquivo muito grande para 116250 (OD)
⚠️ Arquivo muito grande para 116250 (OS)
⚠️ Arquivo muito grande para 15177 (OD)
⚠️ Arquivo muito grande para 15177 (OS)
⚠️ Exame não encontrado para 99003 (OD)
⚠️ Exame não encontrado para 99003 (OS)
⚠️ Arquivo muito grande para 58065 (OD)
⚠️ Arquivo muito grande para 1300 (OD)
⚠️ Arquivo muito grande para 1300 (OS)
⚠️ Arquivo muito grande para 32063 (O

In [59]:
print(df_pentacam.head())

    pront                        nome sexo olho_operado     k1 eixo_plano  \
0  121476      ADEMIR MARTINS PEIXOTO    M           OD  42.92       92.0   
1  114679       ADRIANA DIAS DA CUNHA    F           OD  40.67          3   
2   71580  ALAIDE PEREIRA LIMA AGUIAR    F           OD  44.59        106   
3   65154    ALAOR DE OLIVEIRA PINHAL    M           OD    NaN        NaN   
4   65154    ALAOR DE OLIVEIRA PINHAL    M           OS    NaN        NaN   

      k2 eixo_curvo     al   acd  ... posterior_rp posterior_rc  posterior_rm  \
0  46.72        2.0  22.67  2.98  ...         6.24         6.18          6.21   
1  42.75         93  24.57  2.94  ...         7.16         6.71          6.93   
2  45.62         16  23.59  3.53  ...         6.24         6.12          6.18   
3    NaN        NaN    NaN   NaN  ...         6.31         6.21          6.26   
4    NaN        NaN    NaN   NaN  ...         6.27         6.25          6.26   

   posterior_k1  posterior_k2  posterior_km poster

In [60]:
df_pentacam.to_excel("C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/Pacientes_Torica_com_dados_biometria_atualizados.xlsx", index=False)